# Trabajo 1 - Econometría II 2026-1S
## Modelación ARIMA del Precio Internacional del Aluminio

**Integrantes:** Santiago Vargas, Dilan Tovar, Maria Orjuela

**Profesores:** Milena Hoyos, Luis Luna, German Rodriguez

**Universidad Nacional de Colombia - Facultad de Ciencias Económicas**

---

### Descripción

Este notebook aplica la metodología Box-Jenkins para modelar el **precio internacional del aluminio** (USD por tonelada métrica), con datos del **Fondo Monetario Internacional (FMI)** para el período marzo 2016 a marzo 2026 (121 observaciones mensuales).

### Motivación económica

El aluminio es uno de los metales industriales más utilizados a nivel global. Su importancia económica radica en su papel central en sectores estratégicos:

- **Construcción e infraestructura**: edificios, ventanería, estructuras.
- **Transporte**: automotriz, aeroespacial, ferroviario.
- **Envasado y empaques**: latas, papel aluminio, contenedores.
- **Transición energética**: paneles solares, redes eléctricas, vehículos eléctricos (donde reemplaza al acero por su menor peso).

A diferencia de otros metales industriales, el aluminio tiene una **dinámica de precios bien comportada** que lo hace particularmente adecuado para el modelado ARIMA: presenta tendencia, volatilidad moderada y ausencia de estacionalidad anual marcada. Modelar su evolución resulta relevante para la planeación de costos de manufactura, decisiones de inversión y análisis del ciclo industrial global.

### Estructura del notebook

0. **Configuración** — Importación de librerías y rutas relativas.
1. **Datos** — Lectura del CSV del FMI y preparación de la serie.
2. **Análisis exploratorio** — Visualización y transformación logarítmica.
3. **Identificación** — FAC, FACP y pruebas de raíz unitaria.
4. **Estimación** — Ajuste de modelos ARIMA candidatos.
5. **Validación** — Diagnóstico de residuos.
6. **Pronóstico** — Proyección a 10 meses.

### Fuente de los datos

Fondo Monetario Internacional (FMI), Primary Commodity Prices, serie *Global price of Aluminum* (PALUMUSDM), obtenida vía FRED (Federal Reserve Bank of St. Louis). Unidades: dólares estadounidenses por tonelada métrica, no ajustado estacionalmente.

Archivo original: `data/raw/PALUMUSDM.csv`

## Bloque 0: Configuración

Importación de paquetes y definición de rutas relativas usando `pathlib`. Esto garantiza que el código corra en cualquier computador sin modificar manualmente las rutas.

In [1]:
# %% Importación de paquetes ============================

# Trabajar con rutas relativas en python
from pathlib import Path

# Módulos de numpy, pandas, matplotlib y scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import jarque_bera, probplot, norm

# Módulos de statsmodels
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

# Configuración estética de matplotlib
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Paquetes importados correctamente.")

Paquetes importados correctamente.


In [2]:
# %% Definición de rutas relativas =====================

notebook_path = Path.cwd()
if notebook_path.name == "notebooks":
    PROJECT_ROOT = notebook_path.parent
else:
    PROJECT_ROOT = notebook_path

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

for d in [DATA_PROCESSED, FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto:  {PROJECT_ROOT}")
print(f"Datos crudos:       {DATA_RAW}")
print(f"Datos procesados:   {DATA_PROCESSED}")
print(f"Figuras:            {FIGURES}")
print(f"Tablas:             {TABLES}")

Raíz del proyecto:  c:\postergrupo_mds\poster
Datos crudos:       c:\postergrupo_mds\poster\data\raw
Datos procesados:   c:\postergrupo_mds\poster\data\processed
Figuras:            c:\postergrupo_mds\poster\outputs\figures
Tablas:             c:\postergrupo_mds\poster\outputs\tables


## Bloque 1: Carga y preparación de la serie

El archivo del FMI tiene una estructura simple con dos columnas:
- `observation_date`: fecha mensual (primer día de cada mes).
- `PALUMUSDM`: precio del aluminio en USD por tonelada métrica.

Vamos a:
1. Leer el CSV.
2. Renombrar columnas a nombres claros.
3. Construir el índice temporal mensual.
4. Guardar la serie como CSV procesado en `data/processed/aluminio_precio.csv`.

In [3]:
# %% Lectura del CSV del FMI ============================

ruta_csv_raw = DATA_RAW / "PALUMUSDM.csv"
print(f"Leyendo: {ruta_csv_raw}")

df_raw = pd.read_csv(ruta_csv_raw)

print(f"Forma del DataFrame: {df_raw.shape}")
print(f"\nColumnas: {list(df_raw.columns)}")
print(f"\nPrimeras filas:")
print(df_raw.head())
print(f"\nÚltimas filas:")
print(df_raw.tail())

Leyendo: c:\postergrupo_mds\poster\data\raw\PALUMUSDM.csv
Forma del DataFrame: (121, 2)

Columnas: ['observation_date', 'PALUMUSDM']

Primeras filas:
  observation_date    PALUMUSDM
0       2016-03-01  1531.011905
1       2016-04-01  1571.226190
2       2016-05-01  1550.625000
3       2016-06-01  1593.506818
4       2016-07-01  1629.047619

Últimas filas:
    observation_date    PALUMUSDM
116       2025-11-01  2819.061500
117       2025-12-01  2875.897727
118       2026-01-01  3133.982727
119       2026-02-01  3065.188500
120       2026-03-01  3372.952727


In [4]:
# %% Construcción de la serie de tiempo =================

df = df_raw.rename(columns={
    "observation_date": "fecha",
    "PALUMUSDM": "precio_aluminio"
})

df["fecha"] = pd.to_datetime(df["fecha"])

aluminio_serie = df.set_index("fecha")["precio_aluminio"]
aluminio_serie = aluminio_serie.asfreq("MS")

print(f"Observaciones: {len(aluminio_serie)}")
print(f"Rango temporal: {aluminio_serie.index.min().strftime('%Y-%m')} a {aluminio_serie.index.max().strftime('%Y-%m')}")
print(f"\n¿Hay valores faltantes?: {aluminio_serie.isna().sum()}")
print(f"\nPrimeras 5 observaciones:")
print(aluminio_serie.head())
print(f"\nÚltimas 5 observaciones:")
print(aluminio_serie.tail())

Observaciones: 121
Rango temporal: 2016-03 a 2026-03

¿Hay valores faltantes?: 0

Primeras 5 observaciones:
fecha
2016-03-01    1531.011905
2016-04-01    1571.226190
2016-05-01    1550.625000
2016-06-01    1593.506818
2016-07-01    1629.047619
Freq: MS, Name: precio_aluminio, dtype: float64

Últimas 5 observaciones:
fecha
2025-11-01    2819.061500
2025-12-01    2875.897727
2026-01-01    3133.982727
2026-02-01    3065.188500
2026-03-01    3372.952727
Freq: MS, Name: precio_aluminio, dtype: float64


In [5]:
# %% Estadísticas descriptivas =========================

print("Estadísticas descriptivas del precio del aluminio (USD/tonelada):")
print(aluminio_serie.describe().round(2))
print(f"\nValor mínimo: {aluminio_serie.min():.2f} USD/ton ({aluminio_serie.idxmin().strftime('%b %Y')})")
print(f"Valor máximo: {aluminio_serie.max():.2f} USD/ton ({aluminio_serie.idxmax().strftime('%b %Y')})")

Estadísticas descriptivas del precio del aluminio (USD/tonelada):
count     121.00
mean     2202.96
std       439.95
min      1459.93
25%      1853.72
50%      2184.75
75%      2497.61
max      3498.37
Name: precio_aluminio, dtype: float64

Valor mínimo: 1459.93 USD/ton (Apr 2020)
Valor máximo: 3498.37 USD/ton (Mar 2022)


In [6]:
# %% Guardado del CSV procesado ========================

ruta_csv_proc = DATA_PROCESSED / "aluminio_precio.csv"

aluminio_df = aluminio_serie.reset_index()
aluminio_df.columns = ["fecha", "precio_aluminio"]
aluminio_df.to_csv(ruta_csv_proc, index=False)

print(f"CSV procesado guardado en: {ruta_csv_proc}")
print(f"Tamaño: {ruta_csv_proc.stat().st_size} bytes")
print(f"\nPrimeras 3 filas:")
print(aluminio_df.head(3))

CSV procesado guardado en: c:\postergrupo_mds\poster\data\processed\aluminio_precio.csv
Tamaño: 3445 bytes

Primeras 3 filas:
       fecha  precio_aluminio
0 2016-03-01      1531.011905
1 2016-04-01      1571.226190
2 2016-05-01      1550.625000
